# Christ University Faculty Data Scraper

This notebook scrapes faculty data from Christ University's website across multiple pages (0-10), processes and scales the data, and exports it to a CSV file.

In [1]:
import requests
import json
import pandas as pd
import time
from urllib.parse import urljoin
import re
from sklearn.preprocessing import StandardScaler, LabelEncoder
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

Libraries imported successfully!


In [4]:
# Configuration for Christ University Faculty API
BASE_URL = "https://christuniversity.in/php/ajax/ajax_staffs.php"
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
    'Accept': 'application/json, text/javascript, */*; q=0.01',
    'Accept-Language': 'en-US,en;q=0.9',
    'X-Requested-With': 'XMLHttpRequest',
    'Referer': 'https://christuniversity.in/computer-science/faculty',
    'Content-Type': 'application/x-www-form-urlencoded; charset=UTF-8'
}

# Parameters for pagination
START_PAGE = 0
END_PAGE = 10
DELAY_BETWEEN_REQUESTS = 1  # seconds

print(f"Configuration set: Scraping pages {START_PAGE} to {END_PAGE}")

Configuration set: Scraping pages 0 to 10


In [ ]:
# Test the API endpoint to understand the response format
def test_api_endpoint():
    """Test the API endpoint to understand the response format"""
    print("Testing API endpoint...")
    
    # Test payload
    test_payload = {
        'page': 0,
        'limit': 5,  # Small limit for testing
        'search': '',
        'dept': '',
        'campus': ''
    }
    
    try:
        # Test POST request
        response = requests.post(BASE_URL, headers=HEADERS, data=test_payload, timeout=15)
        print(f"Response status: {response.status_code}")
        print(f"Response headers: {dict(response.headers)}")
        print(f"Response content type: {response.headers.get('content-type')}")
        print(f"Response length: {len(response.text)} characters")
        
        if response.status_code == 200:
            print(f"First 500 characters of response:")
            print(response.text[:500])
            print("\n" + "="*50)
            
            # Try to parse as JSON
            try:
                data = response.json()
                print("✓ Response is valid JSON")
                print(f"Data type: {type(data)}")
                if isinstance(data, dict):
                    print(f"Keys in response: {list(data.keys())}")
                    for key, value in data.items():
                        print(f"  {key}: {type(value)} - {len(value) if isinstance(value, (list, dict, str)) else value}")
                elif isinstance(data, list):
                    print(f"List length: {len(data)}")
                    if data:
                        print(f"First item type: {type(data[0])}")
                        if isinstance(data[0], dict):
                            print(f"First item keys: {list(data[0].keys())}")
                
                return data
            except json.JSONDecodeError as e:
                print(f"✗ JSON decode error: {e}")
                return None
        else:
            print(f"✗ HTTP Error: {response.status_code}")
            return None
            
    except Exception as e:
        print(f"✗ Request failed: {e}")
        return None

# Run the test
test_data = test_api_endpoint()

In [5]:
def scrape_faculty_data(start_page=0, end_page=10):
    """
    Scrape faculty data from Christ University's faculty API
    
    Args:
        start_page (int): Starting page number
        end_page (int): Ending page number
    
    Returns:
        list: List of faculty data dictionaries
    """
    all_faculty_data = []
    total_records = 0
    
    for page in range(start_page, end_page + 1):
        print(f"Scraping page {page}...")
        
        # Data payload for POST request
        payload = {
            'page': page,
            'limit': 12,
            'search': '',
            'dept': '',
            'campus': '',
            'faculty_type': '',
            'qualification': ''
        }
        
        try:
            # Make the API request using POST
            response = requests.post(BASE_URL, headers=HEADERS, data=payload, timeout=30)
            response.raise_for_status()
            
            # Print response for debugging
            print(f"  Response status: {response.status_code}")
            print(f"  Response content type: {response.headers.get('content-type', 'unknown')}")
            
            # Try to parse JSON response
            if response.text.strip():
                try:
                    data = response.json()
                    print(f"  JSON parsed successfully")
                except json.JSONDecodeError:
                    # If not JSON, print first 200 chars of response
                    print(f"  Non-JSON response: {response.text[:200]}...")
                    continue
            else:
                print(f"  Empty response received")
                continue
            
            # Check for data in response
            if isinstance(data, dict):
                if 'results' in data and data['results']:
                    faculty_list = data['results']
                    all_faculty_data.extend(faculty_list)
                    
                    # Update total records count
                    if 'total' in data:
                        total_records = int(data['total'])
                    
                    print(f"  ✓ Page {page}: Found {len(faculty_list)} faculty members")
                elif 'data' in data and data['data']:
                    faculty_list = data['data']
                    all_faculty_data.extend(faculty_list)
                    print(f"  ✓ Page {page}: Found {len(faculty_list)} faculty members")
                else:
                    print(f"  ✗ Page {page}: No data found in response")
                    print(f"    Response keys: {list(data.keys()) if isinstance(data, dict) else 'Not a dict'}")
                    if page > 2:  # Stop after a few empty pages
                        break
            elif isinstance(data, list):
                if data:
                    all_faculty_data.extend(data)
                    print(f"  ✓ Page {page}: Found {len(data)} faculty members")
                else:
                    print(f"  ✗ Page {page}: Empty list received")
                    if page > 2:
                        break
            else:
                print(f"  ✗ Page {page}: Unexpected data format: {type(data)}")
                continue
                
        except requests.exceptions.RequestException as e:
            print(f"  ✗ Page {page}: Request failed - {e}")
            continue
        except json.JSONDecodeError as e:
            print(f"  ✗ Page {page}: Invalid JSON response - {e}")
            print(f"    First 200 chars: {response.text[:200]}...")
            continue
        except Exception as e:
            print(f"  ✗ Page {page}: Unexpected error - {e}")
            continue
        
        # Add delay between requests to be respectful
        time.sleep(DELAY_BETWEEN_REQUESTS)
    
    print(f"\nScraping completed!")
    print(f"Total faculty records collected: {len(all_faculty_data)}")
    print(f"Total records available (from API): {total_records}")
    
    return all_faculty_data

# Run the scraping function
faculty_data = scrape_faculty_data(START_PAGE, END_PAGE)

Scraping page 0...
  Response status: 200
  Response content type: text/html; charset=UTF-8
  JSON parsed successfully
  ✓ Page 0: Found 6 faculty members
  Response status: 200
  Response content type: text/html; charset=UTF-8
  JSON parsed successfully
  ✓ Page 0: Found 6 faculty members
Scraping page 1...
Scraping page 1...
  Response status: 200
  Response content type: text/html; charset=UTF-8
  JSON parsed successfully
  ✓ Page 1: Found 6 faculty members
  Response status: 200
  Response content type: text/html; charset=UTF-8
  JSON parsed successfully
  ✓ Page 1: Found 6 faculty members
Scraping page 2...
Scraping page 2...
  Response status: 200
  Response content type: text/html; charset=UTF-8
  JSON parsed successfully
  ✓ Page 2: Found 6 faculty members
  Response status: 200
  Response content type: text/html; charset=UTF-8
  JSON parsed successfully
  ✓ Page 2: Found 6 faculty members
Scraping page 3...
Scraping page 3...
  Response status: 200
  Response content type: tex

In [6]:
def clean_and_process_data(faculty_data):
    """
    Clean and process the scraped faculty data
    
    Args:
        faculty_data (list): Raw faculty data from API
    
    Returns:
        pd.DataFrame: Cleaned and processed DataFrame
    """
    if not faculty_data:
        print("No data to process!")
        return pd.DataFrame()
    
    # Convert to DataFrame
    df = pd.DataFrame(faculty_data)
    
    print(f"Initial data shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    
    # Remove duplicates based on name and department
    initial_count = len(df)
    df = df.drop_duplicates(subset=['name', 'dept'], keep='first')
    print(f"Removed {initial_count - len(df)} duplicate records")
    
    # Clean name field
    df['name'] = df['name'].str.strip()
    df['name_cleaned'] = df['name'].str.replace(r'^(Dr|Prof|Mr|Ms|Mrs)\.?\s*', '', regex=True)
    
    # Clean department field
    df['dept'] = df['dept'].str.strip()
    df['dept_cleaned'] = df['dept'].str.replace(r'\s*\([^)]*\)', '', regex=True)
    
    # Clean specialization field
    df['spec'] = df['spec'].str.strip()
    
    # Extract additional features from URL
    df['faculty_id'] = df['url'].str.extract(r'faculty-details\/([^\/]+)\/')
    
    # Extract image ID from image URL
    df['image_id'] = df['img'].str.extract(r'E(\d+)\.jpg')
    
    # Create title feature
    df['title'] = df['name'].str.extract(r'^(Dr|Prof|Mr|Ms|Mrs)\.?')
    df['title'] = df['title'].fillna('Unknown')
    
    # Check for missing values
    print("\nMissing values:")
    print(df.isnull().sum())
    
    return df

# Process the scraped data
if faculty_data:
    df_faculty = clean_and_process_data(faculty_data)
    print(f"\nProcessed data shape: {df_faculty.shape}")
    print("\nFirst 3 records:")
    print(df_faculty.head(3).to_string())
else:
    print("No data was scraped to process.")

Initial data shape: (66, 5)
Columns: ['img', 'name', 'dept', 'spec', 'url']
Removed 60 duplicate records

Missing values:
img             0
name            0
dept            0
spec            0
url             0
name_cleaned    0
dept_cleaned    0
faculty_id      0
image_id        0
title           0
dtype: int64

Processed data shape: (6, 10)

First 3 records:
                                                                           img                   name                 dept                          spec                                                         url        name_cleaned   dept_cleaned faculty_id image_id title
0  https://kp.christuniversity.in/KnowledgePro/images/EmployeePhotos/E7062.jpg        Dr AKSHAY KUMAR  MEDIA STUDIES (NCR)            Conflict Reporting  https://christuniversity.in//faculty-details/NzA2Mg==/MA==        AKSHAY KUMAR  MEDIA STUDIES   NzA2Mg==     7062    Dr
1  https://kp.christuniversity.in/KnowledgePro/images/EmployeePhotos/E6400.jpg  Dr HOIMA

In [7]:
def scale_and_encode_data(df):
    """
    Scale numerical features and encode categorical features
    
    Args:
        df (pd.DataFrame): Cleaned faculty DataFrame
    
    Returns:
        pd.DataFrame: Scaled and encoded DataFrame
    """
    if df.empty:
        return df
    
    df_scaled = df.copy()
    
    # Initialize encoders and scalers
    label_encoders = {}
    scaler = StandardScaler()
    
    # Categorical columns to encode
    categorical_columns = ['dept_cleaned', 'spec', 'title']
    
    # Encode categorical variables
    for col in categorical_columns:
        if col in df_scaled.columns:
            le = LabelEncoder()
            # Handle missing values by filling with 'Unknown'
            df_scaled[f'{col}_encoded'] = le.fit_transform(df_scaled[col].fillna('Unknown'))
            label_encoders[col] = le
            print(f"Encoded {col}: {len(le.classes_)} unique categories")
    
    # Create numerical features for scaling
    numerical_features = []
    
    # Name length
    df_scaled['name_length'] = df_scaled['name'].str.len()
    numerical_features.append('name_length')
    
    # Department length
    df_scaled['dept_length'] = df_scaled['dept'].str.len()
    numerical_features.append('dept_length')
    
    # Image ID as numerical (if available)
    if 'image_id' in df_scaled.columns:
        df_scaled['image_id_num'] = pd.to_numeric(df_scaled['image_id'], errors='coerce').fillna(0)
        numerical_features.append('image_id_num')
    
    # Count of words in name
    df_scaled['name_word_count'] = df_scaled['name'].str.split().str.len()
    numerical_features.append('name_word_count')
    
    # Scale numerical features
    if numerical_features:
        df_scaled[numerical_features] = scaler.fit_transform(df_scaled[numerical_features])
        print(f"Scaled numerical features: {numerical_features}")
    
    print(f"\nFinal scaled data shape: {df_scaled.shape}")
    
    # Store encoders and scaler info for reference
    encoding_info = {
        'label_encoders': {col: list(le.classes_) for col, le in label_encoders.items()},
        'numerical_features': numerical_features,
        'scaler_stats': {
            'mean': scaler.mean_.tolist() if hasattr(scaler, 'mean_') else [],
            'scale': scaler.scale_.tolist() if hasattr(scaler, 'scale_') else []
        }
    }
    
    return df_scaled, encoding_info

# Scale and encode the data
if 'df_faculty' in locals() and not df_faculty.empty:
    df_scaled, encoding_info = scale_and_encode_data(df_faculty)
    
    print("\nEncoding Information:")
    for col, classes in encoding_info['label_encoders'].items():
        print(f"{col}: {len(classes)} categories")
    
    print(f"\nNumerical features scaled: {encoding_info['numerical_features']}")
else:
    print("No processed data available for scaling.")

Encoded dept_cleaned: 2 unique categories
Encoded spec: 6 unique categories
Encoded title: 2 unique categories
Scaled numerical features: ['name_length', 'dept_length', 'image_id_num', 'name_word_count']

Final scaled data shape: (6, 17)

Encoding Information:
dept_cleaned: 2 categories
spec: 6 categories
title: 2 categories

Numerical features scaled: ['name_length', 'dept_length', 'image_id_num', 'name_word_count']


In [8]:
def analyze_faculty_data(df_scaled):
    """
    Perform basic analysis on the faculty data
    
    Args:
        df_scaled (pd.DataFrame): Scaled faculty DataFrame
    """
    if df_scaled.empty:
        print("No data to analyze!")
        return
    
    print("=== FACULTY DATA ANALYSIS ===\n")
    
    # Basic statistics
    print(f"Total Faculty Members: {len(df_scaled)}")
    print(f"Unique Departments: {df_scaled['dept_cleaned'].nunique()}")
    print(f"Unique Specializations: {df_scaled['spec'].nunique()}")
    
    # Department distribution
    print("\n--- Department Distribution (Top 10) ---")
    dept_counts = df_scaled['dept_cleaned'].value_counts().head(10)
    for dept, count in dept_counts.items():
        print(f"{dept}: {count}")
    
    # Title distribution
    print("\n--- Title Distribution ---")
    title_counts = df_scaled['title'].value_counts()
    for title, count in title_counts.items():
        print(f"{title}: {count}")
    
    # Specialization distribution
    print("\n--- Specialization Distribution (Top 10) ---")
    spec_counts = df_scaled['spec'].value_counts().head(10)
    for spec, count in spec_counts.items():
        print(f"{spec}: {count}")
    
    # Statistical summary of numerical features
    numerical_cols = ['name_length', 'dept_length', 'name_word_count']
    available_numerical_cols = [col for col in numerical_cols if col in df_scaled.columns]
    
    if available_numerical_cols:
        print("\n--- Numerical Features Statistics ---")
        print(df_scaled[available_numerical_cols].describe())

# Analyze the scaled data
if 'df_scaled' in locals() and not df_scaled.empty:
    analyze_faculty_data(df_scaled)
else:
    print("No scaled data available for analysis.")

=== FACULTY DATA ANALYSIS ===

Total Faculty Members: 6
Unique Departments: 2
Unique Specializations: 6

--- Department Distribution (Top 10) ---
MEDIA STUDIES: 5
PHYSICAL EDUCATION: 1

--- Title Distribution ---
Dr: 5
Unknown: 1

--- Specialization Distribution (Top 10) ---
Conflict Reporting: 1
Communication and Journalism: 1
Mass Communication: 1
Media Ethics: 1
Media and Environment: 1
Physical Education: 1

--- Numerical Features Statistics ---
        name_length   dept_length  name_word_count
count  6.000000e+00  6.000000e+00         6.000000
mean  -2.868076e-16  3.182639e-15         0.000000
std    1.095445e+00  1.095445e+00         1.095445
min   -9.198662e-01 -2.236068e+00        -1.732051
25%   -6.689936e-01  4.472136e-01         0.000000
50%   -5.435573e-01  4.472136e-01         0.000000
75%    5.226513e-01  4.472136e-01         0.000000
max    1.839732e+00  4.472136e-01         1.732051


In [9]:
def export_to_csv(df_scaled, filename='christ_university_faculty_scaled_data.csv'):
    """
    Export the scaled faculty data to CSV file
    
    Args:
        df_scaled (pd.DataFrame): Scaled faculty DataFrame
        filename (str): Output CSV filename
    """
    if df_scaled.empty:
        print("No data to export!")
        return
    
    try:
        # Create a clean version for export
        export_df = df_scaled.copy()
        
        # Select columns for export (both original and processed)
        columns_to_export = [
            'name', 'name_cleaned', 'dept', 'dept_cleaned', 'spec', 'title',
            'img', 'url', 'faculty_id', 'image_id',
            'name_length', 'dept_length', 'name_word_count'
        ]
        
        # Add encoded columns
        encoded_cols = [col for col in export_df.columns if col.endswith('_encoded')]
        columns_to_export.extend(encoded_cols)
        
        # Add numerical features if they exist
        if 'image_id_num' in export_df.columns:
            columns_to_export.append('image_id_num')
        
        # Filter available columns
        available_columns = [col for col in columns_to_export if col in export_df.columns]
        
        # Export to CSV
        export_df[available_columns].to_csv(filename, index=False, encoding='utf-8')
        
        print(f"✓ Data exported to '{filename}' successfully!")
        print(f"  - Rows: {len(export_df)}")
        print(f"  - Columns: {len(available_columns)}")
        print(f"  - File size: {len(export_df) * len(available_columns)} data points")
        
        # Save encoding information as JSON
        info_filename = filename.replace('.csv', '_encoding_info.json')
        with open(info_filename, 'w') as f:
            json.dump(encoding_info, f, indent=2)
        print(f"✓ Encoding information saved to '{info_filename}'")
        
        return available_columns
        
    except Exception as e:
        print(f"✗ Error exporting data: {e}")
        return None

# Export the final processed data
if 'df_scaled' in locals() and not df_scaled.empty:
    exported_columns = export_to_csv(df_scaled)
    
    if exported_columns:
        print(f"\nExported columns: {exported_columns}")
        print(f"\nSample of exported data:")
        print(df_scaled[exported_columns[:5]].head(3).to_string())
else:
    print("No data available for export.")

✓ Data exported to 'christ_university_faculty_scaled_data.csv' successfully!
  - Rows: 6
  - Columns: 17
  - File size: 102 data points
✓ Encoding information saved to 'christ_university_faculty_scaled_data_encoding_info.json'

Exported columns: ['name', 'name_cleaned', 'dept', 'dept_cleaned', 'spec', 'title', 'img', 'url', 'faculty_id', 'image_id', 'name_length', 'dept_length', 'name_word_count', 'dept_cleaned_encoded', 'spec_encoded', 'title_encoded', 'image_id_num']

Sample of exported data:
                    name        name_cleaned                 dept   dept_cleaned                          spec
0        Dr AKSHAY KUMAR        AKSHAY KUMAR  MEDIA STUDIES (NCR)  MEDIA STUDIES            Conflict Reporting
1  Dr HOIMAWATI TALUKDAR  HOIMAWATI TALUKDAR  MEDIA STUDIES (NCR)  MEDIA STUDIES  Communication and Journalism
2         Dr MOHIT KUMAR         MOHIT KUMAR  MEDIA STUDIES (NCR)  MEDIA STUDIES            Mass Communication


In [10]:
# Summary of the entire workflow
print("=== CHRIST UNIVERSITY FACULTY DATA SCRAPING WORKFLOW SUMMARY ===\n")

if 'faculty_data' in locals() and faculty_data:
    print(f"✓ Data Scraping: Successfully scraped {len(faculty_data)} records from pages {START_PAGE}-{END_PAGE}")
else:
    print("✗ Data Scraping: No data was scraped")

if 'df_faculty' in locals() and not df_faculty.empty:
    print(f"✓ Data Cleaning: Processed {len(df_faculty)} unique faculty records")
else:
    print("✗ Data Cleaning: No data was processed")

if 'df_scaled' in locals() and not df_scaled.empty:
    print(f"✓ Data Scaling: Successfully scaled and encoded data with {df_scaled.shape[1]} features")
else:
    print("✗ Data Scaling: No data was scaled")

if 'exported_columns' in locals() and exported_columns:
    print(f"✓ Data Export: Successfully exported to CSV with {len(exported_columns)} columns")
    print(f"  - Main CSV file: christ_university_faculty_scaled_data.csv")
    print(f"  - Encoding info: christ_university_faculty_scaled_data_encoding_info.json")
else:
    print("✗ Data Export: No data was exported")

print(f"\n🎯 Workflow completed! Check your current directory for the output files.")

=== CHRIST UNIVERSITY FACULTY DATA SCRAPING WORKFLOW SUMMARY ===

✓ Data Scraping: Successfully scraped 66 records from pages 0-10
✓ Data Cleaning: Processed 6 unique faculty records
✓ Data Scaling: Successfully scaled and encoded data with 17 features
✓ Data Export: Successfully exported to CSV with 17 columns
  - Main CSV file: christ_university_faculty_scaled_data.csv
  - Encoding info: christ_university_faculty_scaled_data_encoding_info.json

🎯 Workflow completed! Check your current directory for the output files.
